In [ ]:
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

In [ ]:
%%time
spark = create_spark_session(
    aws_profile="default"
)

ev = teehr.RemoteReadWriteEvaluation(spark=spark)

#### Delete NRDS LSTM timeseries (short and medium range) and configuration names:
- nrds_v22_lstm_short_range
- nrds_v22_lstm_medium_range

In [ ]:
%%time
sdf = ev._write.delete_from(
    table_name="secondary_timeseries",
    filters=["configuration_name = 'nrds_v22_lstm_medium_range'"],
    dry_run=False
)

In [ ]:
sdf.count()

In [ ]:
sdf

In [ ]:
%%time
sdf = ev._write.delete_from(
    table_name="configurations",
    filters=["name = 'nrds_v22_lstm_medium_range'"],
    dry_run=False
)

In [ ]:
sdf

In [ ]:
ev.list_tables()

Let's check unique secondary prefixes in the crosswalk

In [ ]:
sdf = ev.spark.sql("""
    SELECT DISTINCT
        SPLIT(secondary_location_id, '-')[0] AS prefix
    FROM iceberg.teehr.location_crosswalks
    WHERE secondary_location_id LIKE '%-%'
""")
sdf.show(truncate=False)

In [ ]:
ev.configurations.to_sdf().show(truncate=False)

Check unique configuration names in the secondary table

In [ ]:
sdf = ev.spark.sql("""
    SELECT DISTINCT configuration_name
    FROM iceberg.teehr.secondary_timeseries
""")
sdf.show(truncate=False)